# The map study (20 September to 3 October 2026): every version, re-scored

Companion to `logs/MAP_ABLATION_LOG.md` (the short note). It re-scores the stored runs of every asset-generation arm of
the third study (the occurrence profile and the relationship map, built by code and given to the LLM or applied after
it), reads each arm's pre-registered rule against the numbers, and re-derives the map's own accuracy check.

- **Runs:** `runs/assets_tuning18_<version>_r<k>/` (gpt-5-mini and gpt-5.4), `runs/assets_heldout26_<version>_r<k>/`,
  `runs/assets_opt_*` (Claude agents). Kept locally; the public repository does not publish them.
- **Reference:** the LAsset authors' manual list (`data/ground_truth/manual_gt_neorv32.json`): 15 tuning modules, 111
  entries; 26 held-out modules, 189 entries. Scored on the modules common to every run of a table.
- **Scorer:** `src/eval_assets.py`, strict. Precision = hits / listed elements. Recall = hits / reference entries. "Paired"
  = per-module difference between an arm and its baseline, modules as the pairing unit, 95% bootstrap interval
  (10,000 draws, seed 0). An interval that includes zero is a null at this sample size.
- **No API call** unless `RUN_API` is set in the first cell; then the last section regenerates one version exactly as the
  record notebook did.
- Nothing is written: the citation checker runs with `write=False`, so the stored trace reports stay as they are.

In [1]:
RUN_API = False      # True: the last section regenerates VERSION_TO_RUN with the OpenAI API. Nothing else calls an API.

import sys, os, io, json, re, hashlib, contextlib, statistics as st
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd

def _repo_root():
    for q in [Path.cwd(), *Path.cwd().parents]:
        if (q / "FINAL_NOTEBOOK.ipynb").exists():
            return q
    raise SystemExit("start this notebook inside the repository")

ROOT = _repo_root()
os.chdir(ROOT)
for _p in ("src", "assetgen_meta", "assetgen_meta/hand_arms", "assetgen_meta/prompt_opt", "step1", "step2", "map_pipeline"):
    sys.path.insert(0, str(ROOT / _p))
import eval_assets as EV, meta_tools as mt, rtl_parse

pd.set_option("display.width", 230); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 80)
REFS = EV.load_refs()
GT = REFS["gt"]
CLOSED = EV.load_closed()

def quiet(fn, *a, **k):
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*a, **k)

def lasset_rtl_only(mods):
    f = ROOT / "data/LAsset_initial_results/asset_list_neorv32_initial.json"
    run = {ip["IP"]: [(a.get("Entity", ""), a.get("Asset RTL", ""), "") for a in ip.get("Assets", []) if a.get("Asset RTL")]
           for ip in json.loads(f.read_text(encoding="utf-8"))}
    return {m: run.get(m, []) for m in mods}

TUNING = sorted(m for m in GT if (ROOT / "data/RTL_data" / f"{m}.vhd").exists())
HELDOUT = sorted(m for m in GT if (ROOT / "data/RTL_heldout" / f"{m}.vhd").exists())
print(f"repo root: {ROOT}")
print(f"manual reference: {len(GT)} modules, {sum(len(v) for v in GT.values())} entries; "
      f"tuning {len(TUNING)} modules / {sum(len(GT[m]) for m in TUNING)} entries; held-out {len(HELDOUT)} / {sum(len(GT[m]) for m in HELDOUT)}")

repo root: E:\jobs\ff\test\Prepoison_subset
manual reference: 41 modules, 300 entries; tuning 15 modules / 111 entries; held-out 26 / 189


## 1. The chain of versions, and the prompt each run was made with

Every arm is a hand edit of the 20 September baseline `m7e194es0ism` (a meta-written procedure prompt plus two worked
examples), or of an arm of it. The cell lists each version with its baseline, the one change, the executor and the
input, then hashes the prompt file on disk and compares it with the `system_sha12` the run loop stamped into
`_run_meta.json`. A match means the text on disk is the text that produced the stored runs. The Claude-executor runs of
the optimization loop have no run meta; their prompt hashes are the ones `prompt_opt/HELDOUT_CHECK_PREREG.md` pins.

In [2]:
HA = ROOT / "assetgen_meta/hand_arms"
ARMS = [
    # label, version, baseline, prompt file, executor, input, one change
    ("baseline", "m7e194es0ism",  None,          HA / "m7e194es0ism/exec_prompt.txt",  "gpt-5-mini", "RTL (comments stripped)", "seed procedure prompt + two worked examples (Step 1b winner)"),
    ("Step 2",   "m7e194es0ism",  "m7e194es0ism", None, "code",       "stored lists + map",       "six frozen rules over the LLM map, applied after generation"),
    ("ismr",     "m7e194es0ismr", "m7e194es0ism", HA / "m7e194es0ismr/exec_prompt.txt", "gpt-5-mini", "RTL + LLM-written map (E3)", "a section that reads the map through the four P3164 questions"),
    ("ismc",     "m7e194es0ismc", "m7e194es0ismr", HA / "m7e194es0ismc/exec_prompt.txt", "gpt-5-mini", "RTL + code-written map", "same prompt as ismr; the map's relationships written by code"),
    ("ismcap",   "m7e194es0ismcap", "m7e194es0ism", HA / "m7e194es0ismcap/exec_prompt.txt", "gpt-5-mini", "RTL", "one bullet: a captured input is a setting point"),
    ("ismq",     "m7e194es0ismq", "m7e194es0ism", HA / "m7e194es0ismq/exec_prompt.txt", "gpt-5-mini", "RTL", "the four CIA questions per value, a required negative, an excluded-values list"),
    ("ismrq",    "m7e194es0ismrq", "m7e194es0ismr", HA / "m7e194es0ismrq/exec_prompt.txt", "gpt-5-mini", "RTL + LLM map", "ismr + the same four questions"),
    ("ismd",     "m7e194es0ismd", "m7e194es0ism", HA / "m7e194es0ismd/exec_prompt.txt", "gpt-5-mini", "RTL", "ASSET_DEFINITION rules: deciders primary, transit registers secondary"),
    ("ist",      "m7e194es0ist",  "m7e194es0ism", HA / "m7e194es0ist/exec_prompt.txt",  "gpt-5-mini", "numbered RTL + map with occurrence IDs + flow graph", "eight-part restructure; one occurrence ID and one record cited per element; questions, influence and exclusion lists"),
    ("ist2",     "m7e194es0ist2", "m7e194es0ist", HA / "m7e194es0ist2/exec_prompt.txt", "gpt-5-mini", "traced inputs v2", "ist without the questions and side lists; CIA in a separate call"),
    ("opt v0 (Claude)", "v0", None, ROOT / "assetgen_meta/prompt_opt/v0/exec_prompt.txt", "Claude agent", "traced inputs v2", "the ist2 prompt on another executor (loop baseline)"),
    ("opt v1 (Claude)", "v1", "v0", ROOT / "assetgen_meta/prompt_opt/v1/exec_prompt.txt", "Claude agent", "traced inputs v2", "three edits: stored registers need a use record; no one-clock copies; inputs into unsupplied sub-units set"),
    ("ist2 on gpt-5.4", "m7e194es0ist2_g54", "m7e194es0ist2", HA / "m7e194es0ist2/exec_prompt.txt", "gpt-5.4", "traced inputs v2", "the ist2 prompt on the stronger model (paired baseline of the final)"),
    ("final on gpt-5.4", "m7e194es0opt1_g54", "m7e194es0ist2_g54", ROOT / "assetgen_meta/prompt_opt/v1/exec_prompt.txt", "gpt-5.4", "traced inputs v2", "opt v1 (the three edits) on gpt-5.4"),
]
OPT_STEM = {"v0": "assets_opt", "v1": "assets_opt"}

def run_dirs(version):
    stem = OPT_STEM.get(version, "assets_tuning18")
    return sorted(ROOT.glob(f"runs/{stem}_{version}_r*"))

rows = []
for label, v, base, pf, model, inp, change in ARMS:
    dirs = run_dirs(v)
    metas = [json.loads((d / "_run_meta.json").read_text(encoding="utf-8")) for d in dirs if (d / "_run_meta.json").exists()]
    stored = sorted({m.get("system_sha12") for m in metas})
    disk = mt.sha12(pf.read_text(encoding="utf-8")) if pf and pf.exists() else "-"
    complete = sum(1 for d in dirs if all((d / f"{m}.json").exists() for m in TUNING))
    rows.append({"arm": label, "version": v, "baseline": base or "-", "executor": model, "input": inp,
                 "sha12 (disk)": disk, "sha12 (runs)": ",".join(s for s in stored if s) or "no run meta",
                 "match": ("n/a (code only)" if pf is None else "yes" if stored and all(s == disk for s in stored) else ("n/a" if not stored else "NO")),
                 "runs on disk": len(dirs), "complete": complete,
                 "chars": len(pf.read_text(encoding="utf-8")) if pf and pf.exists() else None,
                 "first run": min((m.get("timestamp", "") for m in metas), default="")[:16], "change": change})
CHAIN = pd.DataFrame(rows)
display(CHAIN.drop(columns=["change"]))
for r in rows:
    print(f"{r['arm']:18s} {r['change']}")
bad = CHAIN[CHAIN["match"] == "NO"]
print("\nevery stored run's prompt is reproduced byte for byte by the file on disk" if bad.empty else f"\nMISMATCH: {bad['version'].tolist()}")

,arm,version,baseline,executor,input,sha12 (disk),sha12 (runs),match,runs on disk,complete,chars,first run
0,baseline,m7e194es0ism,-,gpt-5-mini,RTL (comments stripped),0d0def4c6fe3,0d0def4c6fe3,yes,3,3,103678.0,2026-09-20 10:04
1,Step 2,m7e194es0ism,m7e194es0ism,code,stored lists + map,-,0d0def4c6fe3,n/a (code only),3,3,NaN,2026-09-20 10:04
2,ismr,m7e194es0ismr,m7e194es0ism,gpt-5-mini,RTL + LLM-written map (E3),fe430b81e2c1,fe430b81e2c1,yes,3,3,114084.0,2026-09-30 01:33
3,ismc,m7e194es0ismc,m7e194es0ismr,gpt-5-mini,RTL + code-written map,fe430b81e2c1,fe430b81e2c1,yes,3,3,114084.0,2026-09-30 01:43
4,ismcap,m7e194es0ismcap,m7e194es0ism,gpt-5-mini,RTL,e1c7d09ba571,e1c7d09ba571,yes,3,3,103966.0,2026-10-01 02:02
5,ismq,m7e194es0ismq,m7e194es0ism,gpt-5-mini,RTL,4629c6a2e3a2,4629c6a2e3a2,yes,3,3,106104.0,2026-10-01 04:15
6,ismrq,m7e194es0ismrq,m7e194es0ismr,gpt-5-mini,RTL + LLM map,24a1ac5f1f71,24a1ac5f1f71,yes,3,3,116510.0,2026-10-01 04:29
7,ismd,m7e194es0ismd,m7e194es0ism,gpt-5-mini,RTL,6f74d32530ab,6f74d32530ab,yes,3,3,104315.0,2026-10-01 15:25
8,ist,m7e194es0ist,m7e194es0ism,gpt-5-mini,numbered RTL + map with occurrence IDs + flow graph,fa07ad720438,fa07ad720438,yes,3,3,193492.0,2026-10-01 23:05
9,ist2,m7e194es0ist2,m7e194es0ist,gpt-5-mini,traced inputs v2,e5fe4918c22b,e5fe4918c22b,yes,3,3,178439.0,2026-10-02 02:05


baseline           seed procedure prompt + two worked examples (Step 1b winner)
Step 2             six frozen rules over the LLM map, applied after generation
ismr               a section that reads the map through the four P3164 questions
ismc               same prompt as ismr; the map's relationships written by code
ismcap             one bullet: a captured input is a setting point
ismq               the four CIA questions per value, a required negative, an excluded-values list
ismrq              ismr + the same four questions
ismd               ASSET_DEFINITION rules: deciders primary, transit registers secondary
ist                eight-part restructure; one occurrence ID and one record cited per element; questions, influence and exclusion lists
ist2               ist without the questions and side lists; CIA in a separate call
opt v0 (Claude)    the ist2 prompt on another executor (loop baseline)
opt v1 (Claude)    three edits: stored registers need a use record; no one-clock copi

## 2. What the model was given

Three input regimes. (a) The baseline and the RTL-only arms: `TARGET IP MODULE`, then the RTL with comments stripped
(`meta_tools.build_user`). (b) `ismr` / `ismc`: the same RTL, then a compact view of the module's relationship map
(occurrence lists dropped, IDs turned into line numbers). (c) The traced arms (`ist` onward): one text file per module
(`assetgen_meta/traced_inputs_v2/<split>/<module>.txt`) with the RTL numbered by source line, the map with occurrence
IDs (one `PORT` / `SIGNAL` line per element, then its occurrences, connections and relationship records) and a flow
graph computed from the records. The cell measures the three parts for the watchdog and over the 15 tuning modules, and
prints the section headings of the baseline prompt and of the final prompt.

In [3]:
TIN = ROOT / "assetgen_meta/traced_inputs_v2"
HEAD = re.compile(r"^=== (.+?) ===", re.M)

def parts(text):
    hs = list(HEAD.finditer(text))
    out = {}
    for i, h in enumerate(hs):
        end = hs[i + 1].start() if i + 1 < len(hs) else len(text)
        out[h.group(1).split("(")[0].strip().replace("RELATIONSHIP MAP: ", "map: ").lower()] = end - h.start()
    return out

stem = "neorv32_wdt"
rtl = rtl_parse.strip_comments((ROOT / "data/RTL_data" / f"{stem}.vhd").read_text(encoding="utf-8", errors="ignore"), vhdl=True)
rtl_only = mt.build_user(stem, rtl)
traced = (TIN / "tuning" / f"{stem}.txt").read_text(encoding="utf-8")
pw = parts(traced)
print(f"{stem}: RTL-only user message {len(rtl_only):,} chars; traced input {len(traced):,} chars = " +
      ", ".join(f"{k} {v:,}" for k, v in pw.items()))
tot = Counter(); n = 0
for m in TUNING:
    t = (TIN / "tuning" / f"{m}.txt").read_text(encoding="utf-8"); n += 1
    for k, v in parts(t).items():
        tot[k] += v
    tot["total"] += len(t)
print(f"15 tuning modules, traced inputs: {tot['total']:,} chars; " +
      ", ".join(f"{k} {v / tot['total']:.1%}" for k, v in tot.items() if k != "total"))
rtl_tot = sum(len(mt.build_user(m, rtl_parse.strip_comments((ROOT / "data/RTL_data" / f"{m}.vhd").read_text(encoding="utf-8", errors="ignore"), vhdl=True))) for m in TUNING)
print(f"the same 15 modules as RTL-only messages: {rtl_tot:,} chars (traced inputs are {tot['total'] / rtl_tot:.1f}x)")

def headings(path, limit=None):
    t = path.read_text(encoding="utf-8")
    hs = [s.strip() for s in t.splitlines() if re.match(r"^\s*(\d+\.\s+[A-Z]|[A-Z][A-Z ,/&()-]{6,}$|### WORKED)", s) and len(s.strip()) < 70]
    return len(t), hs[:limit] if limit else hs
for name, p in (("baseline m7e194es0ism", HA / "m7e194es0ism/exec_prompt.txt"), ("final prompt_opt/v1", ROOT / "assetgen_meta/prompt_opt/v1/exec_prompt.txt")):
    n_chars, hs = headings(p)
    print(f"\n{name}: {n_chars:,} chars; sections: " + " | ".join(hs))

neorv32_wdt: RTL-only user message 4,943 chars; traced input 32,592 chars = rtl 5,826, map: elements and records 23,890, map: flow graph 2,845
15 tuning modules, traced inputs: 943,179 chars; rtl 18.6%, map: elements and records 74.6%, map: flow graph 6.8%
the same 15 modules as RTL-only messages: 148,004 chars (traced inputs are 6.4x)

baseline m7e194es0ism: 103,678 chars; sections: DEFINITIONS | 1. Information | 2. Observation | 3. Use-case flows | 4. External information | 5. Asset definition | 6. Security relevance | 7. Evidence sufficiency | 8. Uncertainty | 9. Analysis procedure | 10. Validation | 11. Output representation | EXECUTOR OUTPUT SCHEMA | WORKED EXAMPLES | ### WORKED EXAMPLE 1: omsp_gpio | FINAL OBJECT | ### WORKED EXAMPLE 2: tiny_aes | FINAL OBJECT

final prompt_opt/v1: 180,670 chars; sections: 1. WHAT YOU ARE GIVEN | 2. WHAT AN ASSET IS | 3. PURPOSE AND USE-CASE FLOWS | 4. THE FLOW GRAPH | 5. ESTABLISHED VALUES | 6. ROLES ALONG THE PATH | 7. NEVER REPORTED | 8. CHECK

## 3. Results: every version, mean over its runs

`ablate()` prints one row per version (mean over repeats, with the sample sd of recall and F1), then recall by element
class and false positives by class. `near` counts hits scored by the record-to-field near-match rather than exact
equality. The reference rows are LAsset's own initial lists on the same modules: spec+RTL and RTL-only (the like-for-like
row, since every arm here reads RTL only). The pooled table after it counts hits over all runs of a version.

In [4]:
VERSIONS = [v for _l, v, _b, _p, _m, _i, _c in ARMS if v not in OPT_STEM]
RUNS = EV.collect(list(dict.fromkeys(["v2x3r8"] + VERSIONS)))        # the August reference row first
MODS = EV.common_modules(RUNS, GT)
extra = {"LAsset spec+RTL": {m: v for m, v in REFS["paper"].items() if m in MODS}, "LAsset RTL-only": lasset_rtl_only(MODS)}
print(f"scored modules: {len(MODS)}, reference entries: {sum(len(GT[m]) for m in MODS)}\n")
SCORES = EV.ablate(RUNS, GT, extra=extra, title="map study, every gpt-5-mini / gpt-5.4 version, mean over runs (strict scorer)")

OPT = EV.collect(["v0", "v1"], stem="assets_opt")
OPTH = EV.collect(["v0", "v1"], stem="assets_opt_heldout")
print()
SCORES_OPT = EV.ablate(OPT, GT, title="prompt-optimization loop, Claude executor, tuning (v0 = the ist2 prompt; v1 = + edits A, B, C)")

  [warn] assets_tuning18_m7e194es0ist2_g54_r2 has 17/18 modules -- INCOMPLETE, excluded. Re-run collect() once generation finishes.
scored modules: 15, reference entries: 111

map study, every gpt-5-mini / gpt-5.4 version, mean over runs (strict scorer)
scored on 15 modules common to every run
version       n  emit   TP   FP   FN      P  recall     sd      F1     sd  near
-------------------------------------------------------------------------------
v2x3r8        3   320   95  225   16  0.296   0.853  0.055   0.439  0.015   0.0
m7e194es0ism  3   273   94  179   17  0.344   0.847  0.024   0.489  0.015   0.0
m7e194es0ismr  3   245   86  158   25  0.353   0.778  0.019   0.485  0.008   0.0
m7e194es0ismc  3   252   87  165   24  0.346   0.787  0.065   0.480  0.024   0.0
m7e194es0ismcap  3   259   92  167   19  0.355   0.829  0.009   0.497  0.005   0.0
m7e194es0ismq  3   238   84  154   27  0.352   0.754  0.053   0.480  0.033   0.0
m7e194es0ismrq  3   215   77  138   34  0.360   0.697  0.02


prompt-optimization loop, Claude executor, tuning (v0 = the ist2 prompt; v1 = + edits A, B, C)
scored on 15 modules common to every run
version       n  emit   TP   FP   FN      P  recall     sd      F1     sd  near
-------------------------------------------------------------------------------
v0            2   294   98  196   12  0.335   0.887  0.045   0.486  0.019   0.0
v1            3   279  104  176    7  0.371   0.934  0.014   0.531  0.007   0.0

  recall by element class                   port       signal signal-field
  v0           (ref 47/32/31)         0.904        0.875        0.903     
  v1           (ref 47/32/31)         0.957        0.938        0.925     

  false positives by class                  port       signal signal-field   port-field
  v0                                     27           54          114            0     
  v1                                     28           49           99            0     


In [5]:
def pooled(runs, versions, mods):
    rows = []
    for v in versions:
        if v not in runs:
            rows.append({"version": v, "runs": 0}); continue
        rs = [EV.score(r, GT, True, only=mods) for r in runs[v]]
        T, Fp, Fn = (sum(r[k] for r in rs) for k in ("tp", "fp", "fn"))
        rows.append({"version": v, "runs": len(rs), "per run TP/FP/FN": "; ".join(f"{r['tp']}/{r['fp']}/{r['fn']}" for r in rs),
                     "pooled P": round(T / (T + Fp), 3), "pooled R": round(T / (T + Fn), 3), "mean listed": round((T + Fp) / len(rs), 1),
                     "mean P": round(EV._mean(r["precision"] for r in rs), 3), "mean R": round(EV._mean(r["recall"] for r in rs), 3),
                     "sd P": round(EV._sd(r["precision"] for r in rs), 3), "sd R": round(EV._sd(r["recall"] for r in rs), 3)})
    return pd.DataFrame(rows)
POOLED = pooled(RUNS, list(dict.fromkeys(["v2x3r8"] + VERSIONS)), MODS)
display(POOLED)
display(pooled(OPT, ["v0", "v1"], MODS))

,version,runs,per run TP/FP/FN,pooled P,pooled R,mean listed,mean P,mean R,sd P,sd R
0,v2x3r8,3,88/216/23; 100/230/11; 96/229/15,0.296,0.853,319.7,0.296,0.853,0.007,0.055
1,m7e194es0ism,3,95/179/16; 91/183/20; 96/176/15,0.344,0.847,273.3,0.344,0.847,0.011,0.024
2,m7e194es0ismr,3,88/159/23; 87/158/24; 84/158/27,0.353,0.778,244.7,0.353,0.778,0.005,0.019
3,m7e194es0ismc,3,91/164/20; 92/172/19; 79/159/32,0.346,0.787,252.3,0.346,0.787,0.013,0.065
4,m7e194es0ismcap,3,93/174/18; 92/164/19; 91/164/20,0.355,0.829,259.3,0.355,0.829,0.006,0.009
5,m7e194es0ismq,3,88/145/23; 86/159/25; 77/158/34,0.352,0.754,237.7,0.352,0.754,0.025,0.053
6,m7e194es0ismrq,3,77/135/34; 75/129/36; 80/149/31,0.360,0.697,215.0,0.360,0.697,0.010,0.023
7,m7e194es0ismd,3,97/192/14; 98/178/13; 97/197/14,0.340,0.877,286.3,0.340,0.877,0.013,0.005
8,m7e194es0ist,3,75/133/36; 77/135/34; 78/143/33,0.359,0.691,213.7,0.359,0.691,0.005,0.014
9,m7e194es0ist2,3,86/151/25; 80/156/31; 78/161/33,0.343,0.733,237.3,0.343,0.733,0.019,0.038


,version,runs,per run TP/FP/FN,pooled P,pooled R,mean listed,mean P,mean R,sd P,sd R
0,v0,2,95/196/16; 102/195/9,0.335,0.887,294.0,0.335,0.887,0.012,0.045
1,v1,3,104/171/7; 105/182/6; 102/174/9,0.371,0.934,279.3,0.371,0.934,0.006,0.014


## 4. Each arm against its baseline, and its pre-registered rule

Paired deltas (modules as the pairing unit) on precision, recall and F1. The rule column is the pass rule as the record
registered it before the run (`experiments/02_rtl_structural_analysis/notebooks/assetgen_meta.ipynb`, the `read_*.py` scripts in `assetgen_meta/hand_arms/`,
`prompt_opt/read_final.py`); the verdict is read off these numbers by code. Two bars recur: the **Step 1b bar** (an arm
is adopted only at precision 0.374 or more with recall 0.83 or more, i.e. +0.03 precision over the baseline's 0.344 at
the recall floor) and the **noise bar** (a change under 0.03 is not a result).

The second cell reads the decisive pre-registered readings that are not plain P/R: `ismcap`'s four target inputs,
`ismq` / `ismrq`'s excluded values that name a reference entry, `ismd`'s transit-register false positives, and the
traced arms' citation checks (share of cited references the checker verifies: the occurrence ID belongs to the element,
the record holds at it, and its type fits the role).

In [6]:
def deltas(runs, base, arm):
    return {k: quiet(EV.paired, runs, base, arm, GT, key=k) for k in ("precision", "recall", "f1")}
def fmt(r):
    return f"{r['delta']:+.3f} [{r['lo']:+.3f}, {r['hi']:+.3f}]"
def mean_pr(runs, v, mods):
    rs = [EV.score(r, GT, True, only=mods) for r in runs[v]]
    return EV._mean(r["precision"] for r in rs), EV._mean(r["recall"] for r in rs), EV._mean(r["emit"] for r in rs)

STEP1B = lambda P, R: P >= 0.374 and R >= 0.83
RULES = {
    "m7e194es0ismr":  ("Step 1b bar: P >= 0.374 at R >= 0.83 (else the map adds nothing to generation)", lambda P, R, d: STEP1B(P, R)),
    "m7e194es0ismc":  ("Step 1b bar; read against ismr: a difference is the map's accuracy", lambda P, R, d: STEP1B(P, R)),
    "m7e194es0ismcap": ("recall repair; adoption still needs the Step 1b bar", lambda P, R, d: STEP1B(P, R)),
    "m7e194es0ismq":  ("P and R not below the base by more than 0.03", lambda P, R, d: d["precision"]["delta"] >= -0.03 and d["recall"]["delta"] >= -0.03),
    "m7e194es0ismrq": ("P and R not below the base (ismr) by more than 0.03", lambda P, R, d: d["precision"]["delta"] >= -0.03 and d["recall"]["delta"] >= -0.03),
    "m7e194es0ismd":  ("Step 1b bar, and R1-R3 of read_def_arm hold", lambda P, R, d: STEP1B(P, R)),
    "m7e194es0ist":   ("Step 1b bar AND >= 80% of citations verified; recall guard R >= 0.80", lambda P, R, d: STEP1B(P, R)),
    "m7e194es0ist2":  ("R >= 0.817 and P >= 0.344 (R2); >= 90% verified (R1); adoption: Step 1b bar", lambda P, R, d: R >= 0.817 and P >= 0.344),
    "m7e194es0opt1_g54": ("F1 dP >= +0.03 'supported', 0 < dP < 0.03 'direction only'; F2 dR >= -0.03", lambda P, R, d: d["precision"]["delta"] >= 0.03 and d["recall"]["delta"] >= -0.03),
}
rows = []
pairs = [(v, b) for _l, v, b, _p, _m, _i, _c in ARMS if b and v in RUNS and b in RUNS and v != b] + [("m7e194es0ist2", "m7e194es0ism")]
for v, b in pairs:
    d = deltas(RUNS, b, v)
    P, R, E = mean_pr(RUNS, v, MODS); Pb, Rb, Eb = mean_pr(RUNS, b, MODS)
    rule, test = RULES.get(v, ("(comparison only)", None))
    rows.append({"arm": v, "baseline": b, "runs": f"{len(RUNS[v])}/{len(RUNS[b])}", "P": f"{P:.3f} (base {Pb:.3f})", "R": f"{R:.3f} (base {Rb:.3f})",
                 "listed/run": f"{E:.1f} (base {Eb:.1f})", "d precision": fmt(d["precision"]), "d recall": fmt(d["recall"]), "d F1": fmt(d["f1"]),
                 "rule": rule, "verdict": "" if test is None else ("PASS" if test(P, R, d) else "FAIL")})
VERDICTS = pd.DataFrame(rows)
display(VERDICTS)

# the Claude loop: v1 against v0 (tuning, mean of 3 vs 2 runs); keep rule F1 > v0 mean + 0.03
d = deltas(OPT, "v0", "v1")
P1, R1, _ = mean_pr(OPT, "v1", MODS); P0, R0, _ = mean_pr(OPT, "v0", MODS)
f1 = lambda p, r: 2 * p * r / (p + r)
print(f"Claude loop, tuning: v1 P {P1:.3f} R {R1:.3f} F1 {f1(P1, R1):.3f} vs v0 P {P0:.3f} R {R0:.3f} F1 {f1(P0, R0):.3f}; "
      f"paired dP {fmt(d['precision'])}, dR {fmt(d['recall'])}, dF1 {fmt(d['f1'])}; keep rule (F1 +0.03 over the v0 mean): "
      f"{'PASS' if f1(P1, R1) - f1(P0, R0) > 0.03 else 'FAIL'}")
HM = EV.common_modules(OPTH, GT)
HP = {}
for v in ("v0", "v1"):
    P, R, E = mean_pr(OPTH, v, HM); HP[v] = (P, R)
    print(f"Claude loop, held-out ({len(HM)} modules, {sum(len(GT[m]) for m in HM)} entries), {v}: P {P:.3f} R {R:.3f} listed {E:.0f} (1 run)")
dh = deltas(OPTH, "v0", "v1")
dP, dR = HP["v1"][0] - HP["v0"][0], HP["v1"][1] - HP["v0"][1]
print(f"held-out v1 - v0 (pooled, as HELDOUT_CHECK_PREREG reads it): dP {dP:+.3f} -> H2: "
      f"{'supported' if dP >= 0.03 else ('direction only, within noise' if dP > 0 else 'not supported')}; dR {dR:+.3f} -> {'holds' if dR >= -0.03 else 'fails'}")
print(f"   module-paired view (mean of per-module P): dP {fmt(dh['precision'])}, dR {fmt(dh['recall'])}; the pooled difference is the registered reading")

,arm,baseline,runs,P,R,listed/run,d precision,d recall,d F1,rule,verdict
0,m7e194es0ismr,m7e194es0ism,3/3,0.353 (base 0.344),0.778 (base 0.847),244.7 (base 273.3),"+0.015 [-0.011, +0.043]","-0.028 [-0.104, +0.046]","+0.002 [-0.033, +0.034]",Step 1b bar: P >= 0.374 at R >= 0.83 (else the map adds nothing to generation),FAIL
1,m7e194es0ismc,m7e194es0ismr,3/3,0.346 (base 0.353),0.787 (base 0.778),252.3 (base 244.7),"-0.002 [-0.025, +0.028]","-0.001 [-0.042, +0.040]","-0.005 [-0.030, +0.022]",Step 1b bar; read against ismr: a difference is the map's accuracy,FAIL
2,m7e194es0ismcap,m7e194es0ism,3/3,0.355 (base 0.344),0.829 (base 0.847),259.3 (base 273.3),"+0.018 [-0.008, +0.046]","+0.004 [-0.050, +0.065]","+0.012 [-0.014, +0.040]",recall repair; adoption still needs the Step 1b bar,FAIL
3,m7e194es0ismq,m7e194es0ism,3/3,0.352 (base 0.344),0.754 (base 0.847),237.7 (base 273.3),"+0.013 [-0.012, +0.043]","-0.065 [-0.140, +0.016]","-0.008 [-0.037, +0.022]",P and R not below the base by more than 0.03,FAIL
4,m7e194es0ismrq,m7e194es0ismr,3/3,0.360 (base 0.353),0.697 (base 0.778),215.0 (base 244.7),"+0.017 [-0.013, +0.052]","-0.069 [-0.111, -0.026]","-0.009 [-0.042, +0.024]",P and R not below the base (ismr) by more than 0.03,FAIL
5,m7e194es0ismd,m7e194es0ism,3/3,0.340 (base 0.344),0.877 (base 0.847),286.3 (base 273.3),"+0.000 [-0.016, +0.018]","+0.031 [+0.005, +0.058]","+0.003 [-0.013, +0.021]","Step 1b bar, and R1-R3 of read_def_arm hold",FAIL
6,m7e194es0ist,m7e194es0ism,3/3,0.359 (base 0.344),0.691 (base 0.847),213.7 (base 273.3),"-0.020 [-0.115, +0.061]","-0.138 [-0.243, -0.031]","-0.057 [-0.130, +0.011]",Step 1b bar AND >= 80% of citations verified; recall guard R >= 0.80,FAIL
7,m7e194es0ist2,m7e194es0ist,3/3,0.343 (base 0.359),0.733 (base 0.691),237.3 (base 213.7),"+0.000 [-0.039, +0.043]","+0.068 [+0.015, +0.127]","+0.018 [-0.019, +0.061]",R >= 0.817 and P >= 0.344 (R2); >= 90% verified (R1); adoption: Step 1b bar,FAIL
8,m7e194es0ist2_g54,m7e194es0ist2,2/3,0.407 (base 0.343),0.815 (base 0.733),222.5 (base 237.3),"+0.086 [+0.035, +0.143]","+0.051 [-0.026, +0.132]","+0.083 [+0.032, +0.139]",(comparison only),
9,m7e194es0opt1_g54,m7e194es0ist2_g54,3/2,0.407 (base 0.407),0.820 (base 0.815),224.0 (base 222.5),"+0.006 [-0.025, +0.040]","+0.010 [-0.040, +0.076]","+0.004 [-0.027, +0.038]","F1 dP >= +0.03 'supported', 0 < dP < 0.03 'direction only'; F2 dR >= -0.03",FAIL


Claude loop, tuning: v1 P 0.371 R 0.934 F1 0.531 vs v0 P 0.335 R 0.887 F1 0.486; paired dP +0.058 [+0.010, +0.125], dR +0.030 [+0.000, +0.075], dF1 +0.051 [+0.017, +0.090]; keep rule (F1 +0.03 over the v0 mean): PASS
Claude loop, held-out (26 modules, 189 entries), v0: P 0.310 R 0.899 listed 548 (1 run)
Claude loop, held-out (26 modules, 189 entries), v1: P 0.329 R 0.931 listed 535 (1 run)
held-out v1 - v0 (pooled, as HELDOUT_CHECK_PREREG reads it): dP +0.019 -> H2: direction only, within noise; dR +0.032 -> holds
   module-paired view (mean of per-module P): dP +0.041 [+0.003, +0.098], dR +0.018 [-0.003, +0.051]; the pooled difference is the registered reading


In [7]:
import trace_check as TC, read_def_arm as rda

def slots(version, mods, stem="assets_tuning18", reps=(0, 1, 2)):
    hit, fp = Counter(), Counter()
    for k in reps:
        d = ROOT / f"runs/{stem}_{version}_r{k}"
        if not d.exists():
            continue
        s = EV.score(d, GT, strict=True, only=set(mods))
        for m in mods:
            miss = Counter(s["per_module"][m]["fn"])
            for n, _o in GT[m]:
                if miss[n] > 0: miss[n] -= 1
                else: hit[(m, n)] += 1
            for n in s["per_module"][m]["fp"]: fp[(m, n)] += 1
    return hit, fp

# ismcap: the four target inputs (falsified if <= 4 of 12 run-slots)
TARGET = [("neorv32_debug_dtm", "jtag_tms_i"), ("neorv32_spi", "spi_dat_i"), ("neorv32_sys", "enable_i"), ("neorv32_uart", "uart_rxd_i")]
hw, _ = slots("m7e194es0ism", MODS); ha, _ = slots("m7e194es0ismcap", MODS)
a, w = sum(ha[k] for k in TARGET), sum(hw[k] for k in TARGET)
print(f"ismcap: target inputs {a}/12 run-slots (winner {w}/12); rule: falsified if <= 4 -> {'FALSIFIED' if a <= 4 else 'ok'}")

# ismq / ismrq: compliance and excluded values that name a reference entry (fail if more than 1 per run)
QS = ("confidentiality", "integrity", "availability", "undermined behavior")
for arm in ("m7e194es0ismq", "m7e194es0ismrq"):
    c = Counter(); ans = Counter(); excl_gt = 0
    for k in range(3):
        for m in MODS:
            f = ROOT / f"runs/assets_tuning18_{arm}_r{k}/_nested/{m}.json"
            o = json.loads(f.read_text(encoding="utf-8"))
            for ce in o.get("conceptual assets", []) or []:
                c["concepts"] += 1
                sq = ce.get("security questions") or {}
                c["four answers"] += all(isinstance(sq.get(q), dict) and sq[q].get("answer") in ("yes", "no") for q in QS)
                for q in QS:
                    if isinstance(sq.get(q), dict): ans[(q, sq[q].get("answer"))] += 1
            names = {n for n, _o in GT[m]}
            for e in o.get("excluded values") or []:
                c["excluded"] += 1
                txt = " ".join(str(e.get(x, "")) for x in ("value", "failed", "statement"))
                excl_gt += any(re.search(r"(?<![\w.])" + re.escape(n) + r"(?![\w])", txt) for n in names)
    print(f"{arm}: concepts {c['concepts']}, with all four answers {c['four answers'] / c['concepts']:.0%}; "
          + "; ".join(f"{q} yes {ans[(q, 'yes')]} / no {ans[(q, 'no')]}" for q in QS)
          + f"; excluded values {c['excluded']} ({c['excluded'] / 45:.1f} per module-run), naming a reference entry {excl_gt} "
          f"({excl_gt / 3:.1f} per run; fail if > 1) -> {'FAIL' if excl_gt / 3 > 1 else 'ok'}")

# ismd: R1 false-positive run-slots on the frozen transit list must fall by half; R3 five protected entries stay >= 2 of 3
_, fw = slots("m7e194es0ism", MODS); hd, fd_ = slots("m7e194es0ismd", MODS)
tb, ta = sum(fw[k] for k in rda.TRANSIT), sum(fd_[k] for k in rda.TRANSIT)
prot = {f"{m[8:]}/{n}": hd[(m, n)] for m, n in rda.PROTECTED}
print(f"ismd: R1 transit-list FP slots {tb} -> {ta} (pass if <= {tb // 2}) -> {'PASS' if ta <= 0.5 * tb else 'FAIL'}; R3 protected hits of 3: {prot} -> "
      f"{'PASS' if all(v >= 2 for v in prot.values()) else 'FAIL'}")

# ist / ist2: citation check (today's checker; nothing written), concepts per run, entries lost against ism
for arm, bar in (("m7e194es0ist", 0.80), ("m7e194es0ist2", 0.90)):
    tr = quiet(TC.run, arm, (0, 1, 2), gt=GT, write=False)
    stc = Counter()
    for k in (0, 1, 2): stc.update(tr[k]["summary"]["ref_status"])
    n = sum(stc.values()); ver = stc["verified"] / n
    print(f"{arm}: cited references {n}, verified {stc['verified']} = {ver:.3f} (bar {bar:.2f}) -> {'PASS' if ver >= bar else 'FAIL'}; statuses {dict(stc)}")
hi, _ = slots("m7e194es0ism", MODS); h2, _ = slots("m7e194es0ist2", MODS); ht, _ = slots("m7e194es0ist", MODS)
lost2 = [(m, n) for m in MODS for n, _o in GT[m] if hi[(m, n)] >= 2 and h2[(m, n)] <= 1]
lost1 = [(m, n) for m in MODS for n, _o in GT[m] if hi[(m, n)] >= 2 and ht[(m, n)] <= 1]
cpr = {}
for v in ("m7e194es0ism", "m7e194es0ist", "m7e194es0ist2"):
    tot = 0
    for k in range(3):
        for m in MODS:
            o = json.loads((ROOT / f"runs/assets_tuning18_{v}_r{k}/_nested/{m}.json").read_text(encoding="utf-8"))
            tot += len([c for c in o.get("conceptual assets", []) or [] if isinstance(c, dict)])
    cpr[v] = tot / 3
print(f"concepts per run: {({k[9:]: round(v, 1) for k, v in cpr.items()})}; ist2 R3 (>= 105): {'PASS' if cpr['m7e194es0ist2'] >= 105 else 'FAIL'}")
print(f"reference entries found by ism in >= 2 runs and lost (<= 1 run): ist {len(lost1)}, ist2 {len(lost2)} (ist2 R8: <= 8 -> {'PASS' if len(lost2) <= 8 else 'FAIL'})")

ismcap: target inputs 4/12 run-slots (winner 2/12); rule: falsified if <= 4 -> FALSIFIED


m7e194es0ismq: concepts 331, with all four answers 100%; confidentiality yes 0 / no 331; integrity yes 327 / no 4; availability yes 199 / no 132; undermined behavior yes 76 / no 255; excluded values 22 (0.5 per module-run), naming a reference entry 3 (1.0 per run; fail if > 1) -> ok


m7e194es0ismrq: concepts 353, with all four answers 98%; confidentiality yes 0 / no 353; integrity yes 351 / no 2; availability yes 253 / no 100; undermined behavior yes 135 / no 210; excluded values 18 (0.4 per module-run), naming a reference entry 5 (1.7 per run; fail if > 1) -> FAIL
ismd: R1 transit-list FP slots 33 -> 43 (pass if <= 16) -> FAIL; R3 protected hits of 3: {'imem/rdata': 3, 'cpu_cp_muldiv/mul.prod': 3, 'cpu_cp_cfu/key_mem': 3, 'hwspinlock/lock_q': 3, 'uart/ctrl.baud': 3} -> PASS


m7e194es0ist: cited references 846, verified 687 = 0.812 (bar 0.80) -> PASS; statuses {'verified': 687, 'edge, role unfit': 32, 'occurrence only': 122, 'map gap claimed': 5, 'invalid': 0, 'no citation': 0}


m7e194es0ist2: cited references 951, verified 778 = 0.818 (bar 0.90) -> FAIL; statuses {'verified': 778, 'edge, role unfit': 61, 'occurrence only': 112, 'map gap claimed': 0, 'invalid': 0, 'no citation': 0}
concepts per run: {'ism': 116.3, 'ist': 92.0, 'ist2': 102.3}; ist2 R3 (>= 105): FAIL
reference entries found by ism in >= 2 runs and lost (<= 1 run): ist 20, ist2 16 (ist2 R8: <= 8 -> FAIL)


## 5. Step 2 (21 September): six frozen rules over the map, applied by code after generation

No API call. `step2/RULES.md` froze the rules before scoring: carrier copy, carrier field, port forwarding, clock and
reset inputs, transport records, a whole record beside its own fields. Each reads only `step1/relation_map/<module>.json`
(the first LLM-written map) and the stored lists of the baseline's three runs. Bar, from the roadmap: precision up by
0.10 with recall down by no more than 0.03. R2 was chosen after looking at the tuning modules, which the rule file says.

In [8]:
import apply_rules as S2
S2.main(["m7e194es0ism"])

15 modules, 111 ground-truth entries (exact); relation map: E:\jobs\ff\test\Prepoison_subset\step1\relation_map

=== m7e194es0ism (3 runs)
   before                             P 0.344  R 0.847  items  273.3


   + R1 carrier copy                  P 0.367  R 0.820  items  247.7


   + R2 carrier field                 P 0.431  R 0.802  items  206.7
   + R3 port forwarding               P 0.431  R 0.802  items  206.7


   + R4 clock and reset               P 0.431  R 0.802  items  206.7


   + R5 transport records             P 0.433  R 0.802  items  205.7
   + R6 record beside its fields      P 0.433  R 0.802  items  205.7


   all rules                          precision +0.089, recall -0.045  -> not substantial (bar: +0.10 and no worse than -0.03)


   all rules + majority vote          P 0.455  R 0.829  items 202   (one list from the 3 runs, reported separately)


## 6. Held-out, pre-registered (1 October): the baseline prompt on the 26 unseen modules, with the code levers

`assetgen_meta/HELDOUT_PREREG.md` pinned 18 files and one decision rule before any held-out call. Levers (frozen in
`post_levers.py`): **MV** keep an element listed in 2 of 3 runs; **NONE** drop an element whose trace path is "none" (the
reasoning quotes none of its lines and the map does not confirm its role label); **BF** add back a whole input port whose
value reaches a stored or computed element within two map hops; **GUARD** drop a record field never read in a condition
(derived from the tuning reference, so secondary only). **R1:** adopt MV+NONE+BF if its precision gain over MV has a 95%
module-bootstrap interval above zero and its recall is at least 0.83. The cell runs `map_pipeline/levers41.py`'s self-test first
(the 17 recorded tuning stacks, the stored held-out reading and the gpt-5.4 base rows must reproduce), then recomputes
the reading from the stored held-out runs.

In [9]:
import levers41 as LV
import post_levers as PL
assert LV.selftest(), "levers41 self-test failed: nothing below is reported"
H = LV.levers(PL.WINNER, "heldout")
T15 = LV.levers(PL.WINNER, "tuning")
la_h, la_t = H.lasset(), T15.lasset()
print(f"\ngpt-5-mini baseline m7e194es0ism: held-out {len(H.modules)} modules, {H.score('base')['ref']} entries | tuning {len(T15.modules)} modules, {T15.score('base')['ref']} entries")
print(f"{'stack':18s} {'tuning P':>9s} {'R':>6s} {'held-out P':>11s} {'R':>6s} {'listed':>7s}")
for name in ("base", "MV", "NONE", "BF", "MV+NONE+BF", "GUARD", "MV+NONE+BF+GUARD"):
    t, h = T15.score(name), H.score(name)
    print(f"{name:18s} {t['precision']:9.3f} {t['recall']:6.3f} {h['precision']:11.3f} {h['recall']:6.3f} {h['emit']:7.1f}")
print(f"{'LAsset spec+RTL':18s} {la_t['precision']:9.3f} {la_t['recall']:6.3f} {la_h['precision']:11.3f} {la_h['recall']:6.3f} {la_h['emit']:7d}")
lr = EV.score(lasset_rtl_only(H.modules), GT, True, only=set(H.modules)); lt = EV.score(lasset_rtl_only(T15.modules), GT, True, only=set(T15.modules))
print(f"{'LAsset RTL-only':18s} {lt['precision']:9.3f} {lt['recall']:6.3f} {lr['precision']:11.3f} {lr['recall']:6.3f} {lr['emit']:7d}")
b = H.bootstrap("MV+NONE+BF", "MV")
r1 = b["dP_lo"] > 0 and H.score("MV+NONE+BF")["recall"] >= 0.83
print(f"\nR1 held-out: MV+NONE+BF minus MV, dP {b['dP']:+.3f} [{b['dP_lo']:+.3f}, {b['dP_hi']:+.3f}], dR {b['dR']:+.3f} [{b['dR_lo']:+.3f}, {b['dR_hi']:+.3f}]; "
      f"recall {H.score('MV+NONE+BF')['recall']:.3f} -> {'PASS: adopted' if r1 else 'FAIL'}")
bt = T15.bootstrap("MV+NONE+BF", "MV")
print(f"R1 tuning reference: dP {bt['dP']:+.3f} [{bt['dP_lo']:+.3f}, {bt['dP_hi']:+.3f}]")
rt = H.random_thinning("GUARD")
print(f"R2 GUARD (secondary): P {rt['guard_P']:.3f} R {rt['guard_R']:.3f} against random thinning of equal strength P {rt['rand_P_mean']:.3f} "
      f"[{rt['rand_P_lo']:.3f}, {rt['rand_P_hi']:.3f}] R {rt['rand_R_mean']:.3f}; draws at or above GUARD: {rt['share_rand_P_ge_guard']:.1%} of {rt['draws']}")
for lever in ("NONE", "BF", "GUARD"):
    print(f"R3 {lever} alone, held-out:", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in H.lever_effect(lever).items()})
per = H.score("base")["per_list"]
print("held-out base per run (P, R, listed, TP):", [(round(x[0], 3), round(x[1], 3), x[2], x[3]) for x in per])

  1. post_levers tuning stacks: 17 stacks + LAsset row, all reproduce


  2. stored held-out reading (base, MV, MV+NONE+BF; R1 interval (0.049, 0.025, 0.08)): reproduces


  3. gpt-5.4 base rows vs results41 per-run means: equal
levers41 self-test PASS



gpt-5-mini baseline m7e194es0ism: held-out 26 modules, 189 entries | tuning 15 modules, 111 entries
stack               tuning P      R  held-out P      R  listed
base                   0.344  0.847       0.352  0.852   458.0
MV                     0.357  0.874       0.350  0.857   463.0
NONE                   0.376  0.844       0.386  0.840   411.7
BF                     0.350  0.874       0.359  0.887   467.0
MV+NONE+BF             0.394  0.901       0.399  0.884   419.0
GUARD                  0.451  0.793       0.446  0.801   340.0
MV+NONE+BF+GUARD       0.500  0.847       0.497  0.836   318.0
LAsset spec+RTL        0.737  0.910       0.734  0.862     222
LAsset RTL-only        0.680  0.748       0.730  0.757     196



R1 held-out: MV+NONE+BF minus MV, dP +0.049 [+0.025, +0.080], dR +0.026 [-0.021, +0.075]; recall 0.884 -> PASS: adopted


R1 tuning reference: dP +0.037 [+0.016, +0.058]


R2 GUARD (secondary): P 0.446 R 0.801 against random thinning of equal strength P 0.393 [0.383, 0.404] R 0.707; draws at or above GUARD: 0.0% of 1000
R3 NONE alone, held-out: {'removed': 46.333, 'added': 0.0, 'fp_removed': 44.0, 'tp_lost': 2.333, 'tp_gained': 0.0, 'fp_added': 0.0, 'share_fp_of_removed': 0.95, 'share_tp_of_added': None}
R3 BF alone, held-out: {'removed': 0.0, 'added': 9.0, 'fp_removed': 0.0, 'tp_lost': 0.0, 'tp_gained': 6.667, 'fp_added': 2.333, 'share_fp_of_removed': None, 'share_tp_of_added': 0.741}
R3 GUARD alone, held-out: {'removed': 118.0, 'added': 0.0, 'fp_removed': 108.333, 'tp_lost': 9.667, 'tp_gained': 0.0, 'fp_added': 0.0, 'share_fp_of_removed': 0.918, 'share_tp_of_added': None}
held-out base per run (P, R, listed, TP): [(0.341, 0.852, 472, 161), (0.35, 0.868, 469, 164), (0.365, 0.836, 433, 158)]


## 7. The final prompt on gpt-5.4, and the levers on it (2 to 3 October)

`map_pipeline/results41.py` scores every configuration on the 41 modules (pooled: mean TP, FP, FN per module over the complete
runs, then P and R from the sums), after a 60-check self-test. `prompt_opt/read_final.py` prints the readings fixed
before the gpt-5.4 run: the paired baseline is the ist2 prompt on the same model, and when it has fewer than 3 complete
runs the readings fall back to ist2 on gpt-5-mini and are labelled confounded. The last cell applies the frozen
MV+NONE+BF to the gpt-5.4 runs (not pre-registered for this model; `levers41.py`).

In [10]:
import final_pipeline as FPL, results41 as R41, read_final as rf
ns = FPL.setup(); mods = FPL.modules(ns.ea)
assert R41.selftest(ns, mods, log=lambda *a, **k: None), "results41 self-test failed"
T = R41.score_table(ns, mods)
show = T[["config", "runs_tuning", "runs_heldout"] + [f"{k}_{s}" for s in ("tuning", "heldout", "all41") for k in ("P", "R")] + ["status"]]
print("results41 self-test PASS. Strict scoring, pooled over the complete runs of each split.")
print(show.round(3).to_string(index=False))
for a in ("final prompt, gpt-5.4", "Step-1b gpt-5-mini"):
    print("\n" + "\n".join(R41._fmt_boot(R41.bootstrap_vs(ns, mods, a, "LAsset RTL-only"))))
print()
_ = rf.read("m7e194es0opt1_g54", base="m7e194es0ist2_g54")
# the paired baseline's incomplete third run: which module is missing
miss = [m for m in TUNING if not (ROOT / "runs/assets_tuning18_m7e194es0ist2_g54_r2" / f"{m}.json").exists()]
print(f"\nist2 on gpt-5.4, run 2: modules without an output {miss} (API credits ran out), so the paired reading uses 2 complete runs")

results41 self-test PASS. Strict scoring, pooled over the complete runs of each split.
               config  runs_tuning  runs_heldout  P_tuning  R_tuning  P_heldout  R_heldout  P_all41  R_all41   status
      LAsset RTL-only            1             1     0.680     0.748      0.730      0.757    0.711    0.753 complete
      LAsset spec+RTL            1             1     0.737     0.910      0.734      0.862    0.735    0.880 complete
       LAsset refined            1             1     0.800     0.901      0.791      0.862    0.795    0.877 complete
final prompt, gpt-5.4            3             3     0.406     0.820      0.388      0.877    0.394    0.856 complete
 final prompt, Claude            3             1     0.371     0.934      0.329      0.931    0.343    0.932 complete
  ist2 prompt, Claude            2             1     0.335     0.887      0.310      0.899    0.319    0.895 complete
   Step-1b gpt-5-mini            3             3     0.344     0.847      0.352      0.


Step-1b gpt-5-mini minus LAsset RTL-only
  all41    41 modules, 300 entries: precision 0.349 vs 0.711, dP -0.362 [-0.423, -0.293]; recall 0.850 vs 0.753, dR +0.097 [+0.039, +0.154]
  tuning   15 modules, 111 entries: precision 0.344 vs 0.680, dP -0.336 [-0.437, -0.232]; recall 0.847 vs 0.748, dR +0.099 [+0.012, +0.204]
  heldout  26 modules, 189 entries: precision 0.352 vs 0.730, dP -0.378 [-0.457, -0.286]; recall 0.852 vs 0.757, dR +0.095 [+0.013, +0.163]



   incomplete runs left out: ['runs/assets_tuning18_m7e194es0ist2_g54_r2'] (re-run the generation cell to fill them)


m7e194es0opt1_g54 readings: 15 modules; reference entries per run [111] (final), [111] (baseline m7e194es0ist2); complete runs 3 (final), 3 (baseline)
   ** baseline on another executor: F1-F3 are CONFOUNDED (prompt and executor both differ) **
   F0  final      P 0.407 R 0.820 F1 0.544 emitted/run 224.0 | per run P [0.418, 0.385, 0.418] R [0.784, 0.829, 0.847]
   F0  baseline   P 0.343 R 0.733 F1 0.467 emitted/run 237.3 | per run P [0.363, 0.339, 0.326] R [0.775, 0.721, 0.703]
   F0  claude v1  P 0.371 R 0.934 F1 0.531 emitted/run 279.3 | per run P [0.378, 0.366, 0.37] R [0.937, 0.946, 0.919]   (other executor: context only)
   F1  precision: final - baseline = +0.064 -> supported
   F2  recall:    final - baseline = +0.087 -> holds


   F3  FPs that edits A + B (as shipped) would still remove, mean per run: final 17.3 ([(8, 6), (8, 11), (6, 13)] A,B per run), baseline 15.7 -> not followed
   F4  cpu interrupt and debug ports listed per run (of 5): final [5, 5, 5], baseline [3, 3, 1]


   F5  listed elements with a verified citation, share per run: final [1.0, 0.967, 0.964], baseline [0.844, 0.869, 0.925]

ist2 on gpt-5.4, run 2: modules without an output ['neorv32_wdt'] (API credits ran out), so the paired reading uses 2 complete runs


In [11]:
res = LV.lever_table()
print("frozen MV+NONE+BF on the final gpt-5.4 runs (not pre-registered for this model)")
for split in ("tuning", "heldout", "all41"):
    for name in LV.STACKS:
        s = res[split][name]
        print(f"{split:8s} {name:11s} P {s['precision']:.3f} R {s['recall']:.3f} listed {s['emit']:.1f} / {s['ref']} entries")
    if split != "all41":
        b = res[split]["boot"]
        print(f"{split:8s} MV+NONE+BF - MV: dP {b['dP']:+.3f} [{b['dP_lo']:+.3f}, {b['dP_hi']:+.3f}]; dR {b['dR']:+.3f} [{b['dR_lo']:+.3f}, {b['dR_hi']:+.3f}]; "
              f"share of listings with a quoted line: {res[split]['paths']['quoted']:.1%}")
bh = res["heldout"]["boot"]
print(f"R1 rule on gpt-5.4 held-out: {'would pass' if bh['dP_lo'] > 0 and res['heldout']['MV+NONE+BF']['recall'] >= 0.83 else 'would NOT pass'} "
      f"(interval {'excludes' if bh['dP_lo'] > 0 else 'includes'} zero; recall {res['heldout']['MV+NONE+BF']['recall']:.3f})")
nd = res["heldout"]["none_drops"]
print("what NONE drops on gpt-5.4 held-out, by role label: " + "; ".join(f"{k}: {v['n']} (hits {v['hits']}, storage mixed {v['mixed']})" for k, v in sorted(nd.items(), key=lambda kv: -kv[1]['n'])))

frozen MV+NONE+BF on the final gpt-5.4 runs (not pre-registered for this model)
tuning   base        P 0.407 R 0.820 listed 224.0 / 111 entries
tuning   MV          P 0.409 R 0.829 listed 225.0 / 111 entries
tuning   MV+NONE+BF  P 0.419 R 0.811 listed 215.0 / 111 entries
tuning   MV+NONE+BF - MV: dP +0.010 [-0.012, +0.030]; dR -0.018 [-0.058, +0.022]; share of listings with a quoted line: 25.0%
heldout  base        P 0.389 R 0.877 listed 427.0 / 189 entries
heldout  MV          P 0.394 R 0.884 listed 424.0 / 189 entries
heldout  MV+NONE+BF  P 0.405 R 0.847 listed 395.0 / 189 entries
heldout  MV+NONE+BF - MV: dP +0.011 [-0.004, +0.029]; dR -0.037 [-0.063, -0.015]; share of listings with a quoted line: 18.0%
all41    base        P 0.395 R 0.856 listed 651.0 / 300 entries
all41    MV          P 0.399 R 0.863 listed 649.0 / 300 entries
all41    MV+NONE+BF  P 0.410 R 0.833 listed 610.0 / 300 entries
R1 rule on gpt-5.4 held-out: would NOT pass (interval includes zero; recall 0.847)
what NONE

## 8. The map itself: accuracy against the gold sample, and the map on LAsset's own lists

The relationship records of the stored tuning maps (`step1/lasset_step1/relation_map_code_tuning/b0e767000ec2_codetags/`,
code writer `step1/code_pairs_v2.py`) against the gold sample of `step1/bakeoff/` (120 occurrences, 212 records; written
by gpt-6-astra in two passes and adjudicated, so a consistency check, not human ground truth). Then the stored reading of
the pre-registered evidence-layer test on LAsset's published held-out list (`assetgen_meta/lasset_layer/`, read once on
2 October; not recomputed here).

In [12]:
acc = FPL.relation_accuracy({"maps": {"tuning": ROOT / "step1/lasset_step1/relation_map_code_tuning/b0e767000ec2_codetags"}})
print("code-written map vs gold:", acc)
EL = json.loads((ROOT / "assetgen_meta/lasset_layer/heldout_rtl_only.json").read_text(encoding="utf-8"))
print(f"\nevidence layer on LAsset's RTL-only held-out list (stored reading): H1 AUC {EL['H1']['auc']:.3f} "
      f"[{EL['H1']['lo']:.3f}, {EL['H1']['hi']:.3f}] -> {EL['verdicts']['H1_audit']}; H2 filter: {EL['verdicts']['H2_filter']} "
      f"(dropped {EL['H2']['dropped']}); claim: {EL['verdicts']['claim']}")

code-written map vs gold: {'occurrences': 120, 'gold records': 212, 'typed precision': 0.988, 'precision 95% CI': (0.969, 1.0), 'typed recall': 0.988, 'recall 95% CI': (0.969, 1.0), 'target-only precision': 1.0, 'target-only recall': 1.0, 'correct / wrong / missed (unweighted)': '210 / 2 / 2'}

evidence layer on LAsset's RTL-only held-out list (stored reading): H1 AUC 0.639 [0.495, 0.768] -> does not hold; H2 filter: not testable (dropped 0); claim: audit trail only: the map traces LAsset's items but does not tell its hits from its false positives


## 9. Where the false positives sit (final prompt, gpt-5.4, all 41 modules, 3 runs)

From the audit trail `map_pipeline/fp_trace41/gpt54/ledger.csv` (one row per listing; written by `map_pipeline/fp_trace41.py` from the
stored runs): the citation status of hits against false positives, and hits and false positives by the role the model
gave each element.

In [13]:
L = pd.read_csv(ROOT / "map_pipeline/fp_trace41/gpt54/ledger.csv")
print(f"listings: {len(L)} over {L['run'].nunique()} run folders, {L['module'].nunique()} modules; labels {dict(L['label'].value_counts())}")
ver = L.groupby("label")["citation"].apply(lambda s: (s == "verified").mean())
print("citation verified: " + ", ".join(f"{k} {v:.1%} ({(L[L.label == k].citation == 'verified').sum()}/{(L.label == k).sum()})" for k, v in ver.items()))
role = L.pivot_table(index="role", columns="label", values="element", aggfunc="count", fill_value=0)
role["hit share"] = (role["TP"] / (role["TP"] + role["FP"])).round(3)
display(role.sort_values("FP", ascending=False))
cb = L[L.cited_type == "CLOCKED_BY"]
print(f"listings citing CLOCKED_BY: {len(cb)} ({len(cb) / len(L):.1%}); hits among them {(cb.label == 'TP').sum()}")

listings: 1953 over 6 run folders, 41 modules; labels {'FP': np.int64(1183), 'TP': np.int64(770)}
citation verified: FP 97.1% (1149/1183), TP 99.5% (766/770)


label,FP,TP,hit share
role,,,
stores,724,245,0.253
computes,306,86,0.219
sets,80,211,0.725
exit port,73,228,0.757


listings citing CLOCKED_BY: 948 (48.5%); hits among them 243


## 10. Regenerating a version (off by default)

With `RUN_API = True` and an `API.env` file holding `OPENAI_API_KEY=...` in the repository root, this cell runs
`VERSION_TO_RUN` for 3 runs on `EXEC_MODEL`, exactly as `experiments/02_rtl_structural_analysis/notebooks/assetgen_meta.ipynb` cell 59 did: one call per module,
the traced input of `assetgen_meta/traced_inputs_v2/tuning/` as the user message, the response kept verbatim under
`_raw/`, the prompt's sha stamped in `_run_meta.json`, and a refusal to write into a folder made with a different prompt.
One 3-run arm is 54 calls. gpt-5.4 at the 2 October list price cost about USD 46 for the 78 held-out calls, so a tuning
arm on it is in the tens of dollars; generation is not deterministic, so the stored runs are not reproduced exactly.

In [14]:
VERSION_TO_RUN, EXEC_MODEL = "m7e194es0opt1_g54", "gpt-5.4"
PROMPT_FILE = ROOT / "assetgen_meta/prompt_opt/v1/exec_prompt.txt"      # 802ee9a90d41; the ist2 prompt is hand_arms/m7e194es0ist2/exec_prompt.txt (e5fe4918c22b)

if RUN_API:
    from dotenv import load_dotenv
    from openai import OpenAI
    from concurrent.futures import ThreadPoolExecutor, as_completed
    load_dotenv(ROOT / "API.env")
    client = OpenAI()
    prompt = PROMPT_FILE.read_text(encoding="utf-8")
    chk = mt.check_exec_prompt(prompt, mt.corpus_names(), VERSION_TO_RUN)
    assert chk["ok"], chk["problems"]
    modules = mt.rtl_modules()
    tin = ROOT / "assetgen_meta/traced_inputs_v2/tuning"
    build = lambda stem, rtl: (tin / f"{stem}.txt").read_text(encoding="utf-8")
    meta = {"stage": f"regenerated by experiments/02_rtl_structural_analysis/notebooks/map_ablation_log.ipynb", "executor": EXEC_MODEL, "system_sha12": mt.sha12(prompt)}
    note = "numbered RTL + relationship map (occurrence IDs, resolved connection directions) + code flow graph (assetgen_meta/traced_inputs_v2)"
    with ThreadPoolExecutor(max_workers=3) as pool:
        futs = {pool.submit(mt.run_version, client, VERSION_TO_RUN, rep, prompt, modules, model=EXEC_MODEL, extra_meta=meta,
                            workers=6, build=build, input_note=note): rep for rep in range(3)}
        for f in as_completed(futs):
            s = f.result()
            print(f"{VERSION_TO_RUN} r{futs[f]}: {s['modules_done']}/{len(modules)} modules, tokens {s['tokens']}, validation issues {s['issues']}")
else:
    print(f"RUN_API is False: nothing generated. Set it to True to run {VERSION_TO_RUN} on {EXEC_MODEL} (3 runs, 18 calls each).")

RUN_API is False: nothing generated. Set it to True to run m7e194es0opt1_g54 on gpt-5.4 (3 runs, 18 calls each).
